In [ ]:
import subprocess
import sys

def install_requirements():
    """Install required packages for Colab environment."""
    requirements = [
        "transformers>=4.35.0",
        "torch>=2.0.0",
        "accelerate",
        "datasets",
        "matplotlib",
        "seaborn",
        "scikit-learn"
    ]

    for package in requirements:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# install
install_requirements()

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import numpy as np
from typing import Dict, List, Tuple, Optional
import copy
import seaborn as sns
import json
import os
import re
from datetime import datetime
import gc
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score
from google.colab import drive
drive.mount('/content/drive')

# Colab Environment konfigurieren
os.environ["TOKENIZERS_PARALLELISM"] = "false"
plt.style.use('default')

class QwenLotteryTicketPruner:
    """
    Lottery Ticket Pruner für Qwen3-4B.
    Optimiert für Colab A100 Umgebung mit QA-spezifischer Evaluation.
    """

    def __init__(self, model, pruning_rate: float = 0.2, structured: bool = False,
                 save_checkpoints: bool = True):
        self.model = model
        self.pruning_rate = pruning_rate
        self.structured = structured
        self.save_checkpoints = save_checkpoints
        self.initial_weights = {}
        self.masks = {}
        self.device = next(model.parameters()).device

        # Checkpoint Path
        self.checkpoint_dir = "/content/drive/MyDrive/lottery_tickets_qwen3_qa"
        os.makedirs(self.checkpoint_dir, exist_ok=True)

        self._store_initial_weights()
        self._print_memory_usage("Nach Initialisierung")

    def _print_memory_usage(self, stage: str):
        """Aktuellen GPU RAM Verbrauch ausgeben."""
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            print(f"{stage}: GPU RAM - Allokiert: {allocated:.2f}GB, Reserviert: {reserved:.2f}GB")

    def _store_initial_weights(self):
        """Speichere initiale Gewichte mit RAM Optimierung und Validierung."""
        print("Speichere initiale Gewichte (lottery ticket)...")

        # Nur trainierbare Parameter der transformer layers speichern
        # Überspringe embedding layers um Model Funktionalität zu wahren
        # Wurden diese layer nicht ausgeschlossen, war die Funktionalität des Modells gestört (kryptische Outputs)
        excluded_layers = ['embed_tokens', 'embed_positions', 'lm_head']

        stored_count = 0
        for name, param in self.model.named_parameters():
            if param.requires_grad and len(param.shape) > 1:
                # Überspringe embedding layers und head für QA
                if not any(excluded in name for excluded in excluded_layers):
                    try:
                        # auf CPU speichern um GPU RAM zu sparen
                        self.initial_weights[name] = param.data.cpu().clone()

                        # Initialisiere Maske
                        mask = torch.ones_like(param.data, dtype=torch.float32)
                        self.masks[name] = mask

                        stored_count += 1
                        param_count = param.numel()
                        print(f"  Speichere {name}: {param_count:,} Parameter")

                    except Exception as e:
                        print(f"Fehler beim Speichern von {name}: {e}")
                        continue

        print(f"{stored_count} Parameter Tensors fürs Pruning gespeichert")

        # Validiere Masken
        self._validate_masks()

        if self.save_checkpoints:
            self.save_checkpoint("initial_state")

    def _validate_masks(self):
        """Validierung, dass alle Masken richtig initialisiert wurden."""
        print("Validiere Masken...")

        for name, mask in self.masks.items():
            if not torch.is_tensor(mask):
                print(f"Fehler: {name} Maske ist kein tensor: {type(mask)}")
                continue

            if torch.any(torch.isnan(mask)) or torch.any(torch.isinf(mask)):
                print(f"Fehler: {name} Maske enthält NaN oder Inf Werte")
                # Reset: Maske = 1
                self.masks[name] = torch.ones_like(mask)
                continue

            total_params = mask.numel()
            active_params = (mask > 0).sum().item()
            print(f"  {name}: {active_params:,}/{total_params:,} aktive Parameter")

    def calculate_importance_scores(self, method: str = "magnitude") -> Dict[str, torch.Tensor]:
        """Berechne importance scores fürs Pruning."""
        importance_scores = {}

        for name, param in self.model.named_parameters():
            if name in self.masks:
                if method == "magnitude":
                    scores = torch.abs(param.data)
                elif method == "gradient" and param.grad is not None:
                    scores = torch.abs(param.grad)
                else:
                    scores = torch.abs(param.data)  # fallback zu magnitude pruning

                importance_scores[name] = scores

        return importance_scores

    def prune_global_magnitude(self):
        """Speicherarmes globales magnitude pruning mit Error Handling."""
        print(f"Global magnitude pruning (rate: {self.pruning_rate})")

        # Schritt 1: Berechne Statistiken mit Validierung
        total_unpruned = 0
        layer_stats = {}

        print("Zähle ungeprunte Gewichte pro layer...")
        for name, param in self.model.named_parameters():
            if name in self.masks:
                try:
                    current_mask = self.masks[name]

                    # Validiere Maske
                    if current_mask.dtype != torch.bool and not torch.is_floating_point(current_mask):
                        print(f"Warnung: Ungültiger Masken dtype für {name}: {current_mask.dtype}")
                        continue

                    # Zähle ungeprunte Gewichte
                    if torch.is_floating_point(current_mask):
                        unpruned_count = (current_mask > 0).sum().item()
                    else:
                        unpruned_count = current_mask.sum().item()

                    # Validiere Anzahl
                    if not isinstance(unpruned_count, (int, float)) or unpruned_count < 0:
                        print(f"Warnung: Ungültige Anzahl für {name}: {unpruned_count}")
                        continue

                    if unpruned_count > 0:
                        layer_stats[name] = int(unpruned_count)
                        total_unpruned += int(unpruned_count)
                        print(f"  {name}: {unpruned_count:,} ungeprunte Gewichte")

                except Exception as e:
                    print(f"Fehler bei der Verarbeitung von Layer {name}: {e}")
                    continue

        print(f"Gesamtzahl ungeprunter Gewichte: {total_unpruned:,}")

        # Validiere Gesamtzahl
        if total_unpruned <= 0 or not isinstance(total_unpruned, int):
            print(f"Fehler: Ungültige Zahl für total_unpruned: {total_unpruned}")
            return

        if total_unpruned > 1e10:  # Sanity check für große Zahlen
            print(f"Fehler: Verdächtig große Anzahl an Gewichten: {total_unpruned:,}")
            return

        num_to_prune = int(total_unpruned * self.pruning_rate)
        print(f"Ziel: Prune {num_to_prune:,} Gewichte von {total_unpruned:,} Gesamt")

        if num_to_prune <= 0:
            print("Es gibt nichts zum prunen!")
            return

        # Schritt 2: Quantilsschätzung
        # Stichprobenergebnisse um globalen Threshold zu schätzen
        sample_scores = []
        sample_size = min(100000, total_unpruned // 20)  # Reduzierte Sample size

        if sample_size <= 0:
            print("Sample Size zu klein!")
            return

        print(f"Sample von {sample_size:,} Gewichten um threshold zu schätzen...")
        sampled_count = 0

        try:
            for name, param in self.model.named_parameters():
                if name in self.masks and name in layer_stats:
                    current_mask = self.masks[name]
                    if (current_mask > 0).sum() > 0:
                        scores = torch.abs(param.data)
                        unpruned_scores = scores[current_mask > 0]

                        # Stichprobe aus dieser Schicht
                        layer_sample_size = min(len(unpruned_scores),
                                              max(1, int(sample_size * layer_stats[name] / total_unpruned)))

                        if layer_sample_size > 0:
                            if len(unpruned_scores) > layer_sample_size:
                                indices = torch.randperm(len(unpruned_scores))[:layer_sample_size]
                                sampled = unpruned_scores[indices]
                            else:
                                sampled = unpruned_scores

                            sample_scores.append(sampled.cpu())  # auf CPU schieben
                            sampled_count += len(sampled)

            if not sample_scores or sampled_count == 0:
                print("Sample inkonsistent oder leer! Fallback zu layerweisem Pruning.")
                self._fallback_layerwise_pruning()
                return

            # Berechne threshold aus Stichprobe
            print(f"Berechne threshold aus {sampled_count:,} gezogenen Gewichten...")
            all_samples = torch.cat(sample_scores)

            # Validiere Stichprobe
            if torch.any(torch.isnan(all_samples)) or torch.any(torch.isinf(all_samples)):
                print("Warnung: NaN oder Inf Werte in Stichprobe, bereinige...")
                all_samples = all_samples[torch.isfinite(all_samples)]

            if len(all_samples) == 0:
                print("Keine valide Stichprobe! Fallback zu layerweisem Pruning.")
                self._fallback_layerwise_pruning()
                return

            percentile = self.pruning_rate * 100
            threshold = torch.quantile(all_samples, percentile / 100.0).item()

            print(f"Geschätzter Threshold: {threshold:.6f}")

        except Exception as e:
            print(f"Fehler während der Stichprobe: {e}")
            print("Fallback zu layerweisem Pruning...")
            self._fallback_layerwise_pruning()
            return

        # Schritt 3: Pruning Layer by Layer
        pruned_count = 0

        try:
            for name, param in self.model.named_parameters():
                if name in self.masks:
                    current_mask = self.masks[name]
                    if (current_mask > 0).sum() > 0:
                        scores = torch.abs(param.data)

                        # Finde Gewichte zum Prunen
                        to_prune = (scores <= threshold) & (current_mask > 0)
                        layer_pruned = to_prune.sum().item()

                        # Wende Pruning an
                        current_mask[to_prune] = 0

                        pruned_count += layer_pruned

                        if layer_pruned > 0:
                            print(f"  {name}: {layer_pruned:,} geprunte Gewichte")

            print(f"Gesamtzahl geprunt: {pruned_count:,} Gewichte (Threshold: {threshold:.6f})")

        except Exception as e:
            print(f"Fehler während des Prunings: {e}")

        finally:
            # RAM aufräumen
            if 'sample_scores' in locals():
                del sample_scores
            if 'all_samples' in locals():
                del all_samples
            torch.cuda.empty_cache()

    def _fallback_layerwise_pruning(self):
        """Fallback zu vereinfachtem layerweisem Pruning, falls globales Pruning fehlschlägt."""
        print("Wende layerweises Pruning an...")

        pruned_count = 0
        for name, param in self.model.named_parameters():
            if name in self.masks:
                current_mask = self.masks[name]
                active_weights = (current_mask > 0).sum().item()

                if active_weights > 0:
                    scores = torch.abs(param.data)
                    unpruned_scores = scores[current_mask > 0]

                    layer_prune_count = int(active_weights * self.pruning_rate)
                    if layer_prune_count > 0:
                        threshold = torch.topk(unpruned_scores, layer_prune_count, largest=False)[0][-1]

                        to_prune = (scores <= threshold) & (current_mask > 0)
                        current_mask[to_prune] = 0

                        actual_pruned = to_prune.sum().item()
                        pruned_count += actual_pruned

                        print(f"  {name}: {actual_pruned:,}/{active_weights:,} geprunte Gewichte")

        print(f"Fallback Pruning abgeschlossen: {pruned_count:,} Gewichte gepruned")

    def apply_masks(self):
        """Wende aktuelle Maske auf die Model Parameter an."""
        for name, param in self.model.named_parameters():
            if name in self.masks:
                param.data *= self.masks[name]

    def reset_to_lottery_ticket(self):
        """Reset auf initiale Gewichte mit aktueller Maske."""
        print("Reset zum lottery ticket...")

        for name, param in self.model.named_parameters():
            if name in self.initial_weights:
                initial_weight = self.initial_weights[name].to(self.device)
                param.data.copy_(initial_weight)
                param.data *= self.masks[name]
                del initial_weight

        torch.cuda.empty_cache()
        self._print_memory_usage("Nach dem lottery ticket reset")

    def get_sparsity_stats(self) -> Dict[str, float]:
        """Berechne detaillierte Sparsity Statistiken."""
        stats = {'layers': {}}
        total_params = 0
        total_pruned = 0

        for name, mask in self.masks.items():
            layer_total = mask.numel()
            layer_pruned = (mask == 0).sum().item()
            layer_sparsity = layer_pruned / layer_total

            stats['layers'][name] = {
                'sparsity': layer_sparsity,
                'remaining': layer_total - layer_pruned,
                'total': layer_total
            }

            total_params += layer_total
            total_pruned += layer_pruned

        stats['overall_sparsity'] = total_pruned / total_params
        stats['total_params'] = total_params
        stats['remaining_params'] = total_params - total_pruned

        return stats

    def save_checkpoint(self, checkpoint_name: str):
        """Speicher aktuellen State."""
        if not self.save_checkpoints:
            return

        checkpoint_path = os.path.join(self.checkpoint_dir, f"{checkpoint_name}.pt")

        checkpoint = {
            'masks': {name: mask.cpu() for name, mask in self.masks.items()},
            'sparsity_stats': self.get_sparsity_stats(),
            'pruning_rate': self.pruning_rate,
            'timestamp': datetime.now().isoformat()
        }

        torch.save(checkpoint, checkpoint_path)
        print(f"Checkpoint gespeichert: {checkpoint_path}")

class QwenQALotteryExperiment:
    """
    Lottery ticket experiment for Qwen3-4B.
    """

    def __init__(self, model_path: str, input_json: str):
        self.model_path = model_path
        self.input_json = input_json
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Lade Model und Tokenizer
        self.setup_model_and_tokenizer()
        self.load_dataset()

    def setup_model_and_tokenizer(self):
        """Lade Qwen3-4B Modell und Tokenizer."""
        print(f"Lade Modell von {self.model_path}...")

        # Lade Tokenizer mit Settings
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_path,
            trust_remote_code=True
        )

        # Tokenizer config
        self.tokenizer.padding_side = 'left'
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Lade Modell mit RAM Optimierung
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            torch_dtype=torch.float16,  # FP16 für RAM Effizienz
            device_map="auto",
            trust_remote_code=True
        )

        # Gen Config
        self.gen_conf = GenerationConfig(
            max_new_tokens=200,
            do_sample=False
        )

        print(f"Modell erfolgreich geladen")
        self._print_memory_usage("Nachdem das Modell geladen wurde ...")

    def _print_memory_usage(self, stage: str):
        """Gebe GPU RAM Verbrauch aus."""
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            print(f"{stage}: GPU RAM - Allokiert: {allocated:.2f}GB, Reserviert: {reserved:.2f}GB")

    def load_dataset(self):
        """Lade QA dataset."""
        print(f"Lade dataset von {self.input_json}...")

        with open(self.input_json, 'r', encoding='utf-8') as f:
            data = json.load(f)
            self.questions = data['questions']

        print(f"Loaded {len(self.questions)} questions")

        # Teilen in train/eval für lottery ticket experimente
        split_idx = int(len(self.questions) * 0.8)
        self.train_questions = self.questions[:split_idx]
        self.eval_questions = self.questions[split_idx:]

        print(f"Train: {len(self.train_questions)}, Eval: {len(self.eval_questions)}")

    def build_messages(self, qtext, snippets, qtype, mode="exact"):
        """Prompts bauen."""
        system_msg = {"role":"system","content":"/no_think"}
        ctx = "\n".join(s["text"] for s in snippets[:2])

        if mode == "exact":
            if qtype == "yesno":
                content = f"Question: {qtext}\nContext:\n{ctx}\nAnswer only 'yes' or 'no', in English, no extras."
            elif qtype == "factoid":
                content = f"Question: {qtext}\nContext:\n{ctx}\nProvide up to 5 keywords, comma-separated, in English, no commentary."
            elif qtype == "list":
                content = f"Question: {qtext}\nContext:\n{ctx}\nProvide a comma-separated list of relevant items, in English, no filler words."
            else:
                content = f"Question: {qtext}\nContext:\n{ctx}\nProvide a brief answer in English."
        else:  # ideal
            if qtype == "yesno":
                content = f"Question: {qtext}\nContext:\n{ctx}\nProvide one-sentence ideal answer in English starting with 'Yes,' or 'No,'."
            else:
                content = f"Question: {qtext}\nContext:\n{ctx}\nProvide an ideal answer in English (one paragraph, max 200 words, full sentences)."

        user_msg = {"role":"user","content":content}
        return [system_msg, user_msg]

    def clean_exact(self, text, qtype):
        """Text bereinigen."""
        txt = text.strip()
        txt = re.sub(r'<\/think>','', txt)
        txt = re.sub(r'\s*(Okay\.?|etc\.?|usw\.?|\.\.\.)$', '', txt, flags=re.IGNORECASE)

        if qtype == "yesno":
            return "yes" if txt.lower().startswith("yes") else "no"
        if qtype in ("factoid","list"):
            items = [i.strip() for i in txt.split(",") if i.strip()]
            return items
        return txt

    def clean_ideal(self, text, qtype):
        """Text bereinigen."""
        txt = text.strip()
        txt = re.sub(r'<\/think>','', txt)
        txt = re.sub(r'\s*(Okay\.?|etc\.?|usw\.?|\.\.\.)$', '', txt, flags=re.IGNORECASE)

        sentences = re.split(r'(?<=[.!?])\s+', txt)
        if qtype == "yesno":
            return sentences[0].strip()

        total = 0
        out = []
        for sent in sentences:
            length = len(sent.split())
            if total + length <= 200:
                out.append(sent)
                total += length
            else:
                break
        return " ".join(out).strip()

    def evaluate_qa_performance(self, questions_subset, batch_size=4):
        """Evaluaiere QA Performance."""
        self.model.eval()
        results = []

        with torch.no_grad():
            for i in tqdm(range(0, len(questions_subset), batch_size), desc='Evaluatiere'):
                batch = questions_subset[i:i+batch_size]

                # Exakte Antwort
                msgs_ex = [self.build_messages(q['body'], q.get('snippets',[]), q['type'], mode='exact') for q in batch]
                texts_ex = [self.tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False) for m in msgs_ex]
                inputs_ex = self.tokenizer(texts_ex, return_tensors='pt', padding=True, truncation=True).to(self.model.device)

                out_ex = self.model.generate(**inputs_ex, generation_config=self.gen_conf)

                dec_ex = []
                for idx, q in enumerate(batch):
                    start = inputs_ex['input_ids'].shape[1]
                    ids = out_ex[idx][start:].tolist()
                    text = self.tokenizer.decode(ids, skip_special_tokens=True)
                    dec_ex.append(self.clean_exact(text, q['type']))

                # Speichere results
                for idx, q in enumerate(batch):
                    results.append({
                        'id': q['id'],
                        'type': q['type'],
                        'exact_answer': q.get('exact_answer'),
                        'exact_prediction': dec_ex[idx],
                        'question': q['body']
                    })

        return results

    def calculate_qa_metrics(self, results):
        """Berechne QA-spezifische Metriken."""
        metrics = {'overall': {}, 'by_type': {}}

        # Gruppiere nach Frage-Typ
        by_type = {}
        for result in results:
            qtype = result['type']
            if qtype not in by_type:
                by_type[qtype] = {'correct': 0, 'total': 0}

            by_type[qtype]['total'] += 1

            # Überprüfe Richtigkeit
            pred = result['exact_prediction']
            truth = result['exact_answer']

            if qtype == "yesno":
                if isinstance(pred, str) and isinstance(truth, str):
                    correct = pred.lower().strip() == truth.lower().strip()
                else:
                    correct = str(pred).lower() == str(truth).lower()
            elif qtype in ["factoid", "list"]:
                if isinstance(pred, list) and isinstance(truth, list):
                    # Berechne overlap für Listen
                    pred_set = set(str(x).lower().strip() for x in pred)
                    truth_set = set(str(x).lower().strip() for x in truth)
                    correct = len(pred_set & truth_set) > 0  # Mindestens ein Match
                else:
                    correct = False
            else:
                # Für alle anderen Fragetypen, einfaches String matching ..
                correct = str(pred).lower().strip() == str(truth).lower().strip()

            if correct:
                by_type[qtype]['correct'] += 1

        # Berechne Metriken nach Fragetyp
        total_correct = 0
        total_questions = 0

        for qtype, stats in by_type.items():
            accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
            metrics['by_type'][qtype] = {
                'accuracy': accuracy,
                'correct': stats['correct'],
                'total': stats['total']
            }
            total_correct += stats['correct']
            total_questions += stats['total']

        # Gesamtmetriken
        metrics['overall'] = {
            'accuracy': total_correct / total_questions if total_questions > 0 else 0,
            'correct': total_correct,
            'total': total_questions
        }

        return metrics

    def run_lottery_ticket_qa_experiment(self,
                                       num_iterations: int = 5,
                                       pruning_rate: float = 0.32,
                                       eval_batch_size: int = 2):  # Reduzierte batch size
        """
        Führe lottery ticket Experiment aus.
        """
        # Initialisiere pruner
        pruner = QwenLotteryTicketPruner(self.model, pruning_rate=pruning_rate)

        # Ergebnisse nachverfolgen
        results = []

        print("\n" + "="*60)
        print("STARTE QA LOTTERY TICKET EXPERIMENT")
        print("="*60)

        for iteration in range(num_iterations):
            print(f"\nITERATION {iteration + 1}/{num_iterations}")
            print("-" * 40)

            # Bereinige cache vor jeder Iteration
            torch.cuda.empty_cache()
            gc.collect()

            # Reset zum lottery ticket (außer bei erster Iteration)
            if iteration > 0:
                pruner.reset_to_lottery_ticket()

            # wende aktuelle Maske an
            pruner.apply_masks()

            # Evaluiere auf kleinerem Dataset aufgrund von Speicher - Schwierigkeiten
            eval_subset = self.eval_questions[:min(50, len(self.eval_questions))]
            print(f"Evaluating on {len(eval_subset)} questions...")

            qa_results = self.evaluate_qa_performance(eval_subset, batch_size=eval_batch_size)
            metrics = self.calculate_qa_metrics(qa_results)
            sparsity_stats = pruner.get_sparsity_stats()

            # Ergebnisse speichern
            result = {
                'iteration': iteration + 1,
                'overall_accuracy': metrics['overall']['accuracy'],
                'yesno_accuracy': metrics['by_type'].get('yesno', {}).get('accuracy', 0),
                'factoid_accuracy': metrics['by_type'].get('factoid', {}).get('accuracy', 0),
                'list_accuracy': metrics['by_type'].get('list', {}).get('accuracy', 0),
                'sparsity': sparsity_stats['overall_sparsity'],
                'remaining_params': sparsity_stats['remaining_params'],
                'total_params': sparsity_stats['total_params'],
                'detailed_metrics': metrics
            }
            results.append(result)

            # Gebe Ergebnisse aus
            print(f"  Results:")
            print(f"  Overall Accuracy: {result['overall_accuracy']:.2%}")
            print(f"  Yes/No Accuracy: {result['yesno_accuracy']:.2%}")
            print(f"  Factoid Accuracy: {result['factoid_accuracy']:.2%}")
            print(f"  List Accuracy: {result['list_accuracy']:.2%}")
            print(f"  Sparsity: {result['sparsity']:.2%}")
            print(f"  Remaining Params: {result['remaining_params']:,}")

            # Speichere Checkpoint
            pruner.save_checkpoint(f"qa_iteration_{iteration + 1}")

            # RAM vor dem Prunen aufräumen
            torch.cuda.empty_cache()
            gc.collect()

            # Pruning Phase (außer für die letzte Iteration)
            if iteration < num_iterations - 1:
                print("Pruning phase...")
                pruner.prune_global_magnitude()

                # zusätzlicher Clean-Up nach dem Pruning
                torch.cuda.empty_cache()
                gc.collect()

        print("\n" + "="*60)
        print("QA EXPERIMENT ABGESCHLOSSEN!")
        print("="*60)

        return self.model, pruner, results

# Haupt Funktion zur Ausführung
def run_qwen3_qa_lottery_experiment():
    # file paths
    MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B"
    INPUT_JSON = "/content/drive/MyDrive/Colab Notebooks/12B_combined_golden.json"

    # Check GPU und definiere RAM Optimierung
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name()
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"GPU Detected: {gpu_name}")
        print(f"GPU Memory: {gpu_memory:.1f} GB")

        # aktivier RAM Optimierung
        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

        if gpu_memory < 20:
            print("Warnung: Weniger als 20GB GPU RAM verfügbar. Benutze nun konservative settings.")
            eval_batch_size = 1
            pruning_rate = 0.05  # niedrigere pruning rate
        else:
            eval_batch_size = 2
            pruning_rate = 0.1  # Mehr konservativ als 0.2
    else:
        print("Keine GPU erkannt, CUDA benötigt!")
        return None

    # Bereinige gecacheten Speicher
    torch.cuda.empty_cache()

    try:
        # Erstelle Experiment
        experiment = QwenQALotteryExperiment(MODEL_PATH, INPUT_JSON)

        # Führe lottery ticket experiment aus
        model, pruner, results = experiment.run_lottery_ticket_qa_experiment(
            num_iterations=5,
            pruning_rate=pruning_rate,
            eval_batch_size=eval_batch_size
        )

        # Gebe finale Zusammenfassung aus
        print("\nFINALE QA LOTTERY TICKET ZUSAMMENFASSUNG")
        print("-" * 50)
        for i, result in enumerate(results):
            print(f"Iteration {i+1}: "
                  f"Sparsity={result['sparsity']:.1%}, "
                  f"Overall Acc={result['overall_accuracy']:.2%}, "
                  f"Yes/No={result['yesno_accuracy']:.2%}, "
                  f"Factoid={result['factoid_accuracy']:.2%}")

        # Speichere Ergebnisse
        results_path = "/content/drive/MyDrive/lottery_tickets_qwen3_qa/detailed_results.json"
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\n💾 Detaillierte Ergebnisse gespeichert: {results_path}")

        return model, pruner, results

    except torch.cuda.OutOfMemoryError as e:
        print(f"CUDA OutOfMemoryError: {e}")

        # Bereinige RAM
        torch.cuda.empty_cache()
        gc.collect()

        return None

# Führe Experiment aus
results = run_qwen3_qa_lottery_experiment()